# Routing

While a sequential workflow, as seen in the Prompt Chaining pattern, is foundational to think about multi-step agent applications, it has several limitations:
- It lacks the ability to make decisions based on context. 
- Without a mechanism to choose the correct tool or sub-process for a specific task, the system remains rigid and non-adaptive. 
- It makes difficult to build sophisticated applications that can manage the complexity and variability of real-world user requests.

The Routing pattern provides a solution by introducing conditional logic, enabling the system to first analyze an incoming query to determine its intent or nature. Based on this analysis, the agent dynamically directs the flow of control to the most appropriate specialized tool, function, or sub-agent.



## Implementation with Flyte V2

This notebook refactors the original LangChain + Langraph routing example into a Flyte v2 task that uses plain Python for the routing logic. The reason this example moves entirely to Flyte V2 is because the delta of what LangGraph offered and what can be done natively on Flyte was too low:

 | | LangGraph | Flyte V2 |                                    
  |---|---|---|                                                 
  | Routing logic | Explicit graph with nodes/edges | Plain Python `if/else` inside a task |                              
  | State | Shared typed dict flowing through nodes | Immutable return values between tasks |                                 
  | Cycles | First-class (loops back to any node) | Implemented as `while` loop inside a task |                               
  | Human-in-the-loop | Built-in graph interrupts | HITL plugin |                                         
  | Streaming | State dict streams between nodes | `@flyte.trace` + async generators |                           
  | Execution model | In-process | Separate pods per task | 
  | Per-task resources/images | No | Yes |
  | Caching | No | Yes |    

1. Install dependencies

In [ ]:
!uv pip install flyte openai

2. Create a secret in Flyte for the API key (only done once):

In [ ]:
!flyte create secret OPENAI_API_KEY --value sk-proj-...

3. Import dependencies and declare the resources your execution environment will need, using the `TaskEnvironment` class. This allows you to allocate precise resources and run agents on their own container with all dependencies baked in automatically:

In [ ]:
import os
import asyncio
import flyte
from flyte import TaskEnvironment, Resources, Secret
from openai import AsyncOpenAI

flyte.init_from_config()

env = TaskEnvironment(
    name="routing_env",
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",
    image=flyte.Image.from_debian_base().with_pip_packages("openai"),
    secrets=[flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY")],
)


## Implementation with Flyte V2

In Flyte V2, **composition happens via plain Python inside tasks** — there are no `@workflow` or `@dynamic` decorators, and no external graph library is needed. The routing pattern maps directly to:

- An `@env.task` that calls the LLM and returns the decision string (`"booker"`, `"info"`, or `"unclear"`)
- Three handler `@env.task` functions, one per route
- A top-level `@env.task` that `await`s the router, then uses a plain `if/else` to call the right handler

Define the router task — it calls the LLM and returns the routing decision:

In [ ]:
@env.task
async def router_node(request: str) -> str:
    """Call the LLM to decide how to route the request. Returns 'booker', 'info', or 'unclear'."""
    client = AsyncOpenAI()
    messages = [
        {
            "role": "system",
            "content": (
                "Analyze the user's request and determine which specialist handler should process it.\n"
                "- If the request is related to booking flights or hotels, output 'booker'.\n"
                "- For general information questions, output 'info'.\n"
                "- If the request is unclear or doesn't fit either category, output 'unclear'.\n"
                "ONLY output one word: 'booker', 'info', or 'unclear'."
            ),
        },
        {"role": "user", "content": request},
    ]
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=messages,
    )
    decision = (resp.choices[0].message.content or "").strip().lower()
    if decision not in ("booker", "info", "unclear"):
        decision = "unclear"
    return decision

Define the handler tasks — each is a specialized `@env.task` that processes a specific type of request:

In [ ]:
@env.task
async def booking_handler(request: str) -> str:
    return f"Booking Handler processed request: '{request}'. Result: Simulated booking action."


@env.task
async def info_handler(request: str) -> str:
    return f"Info Handler processed request: '{request}'. Result: Simulated information retrieval."


@env.task
async def unclear_handler(request: str) -> str:
    return f"Coordinator could not delegate request: '{request}'. Please clarify."

Wire the router and handlers into a single entry-point task. Because Flyte V2 tasks are plain async functions, the routing logic is just a Python `if/else` — no graph compilation or state machine needed:

In [ ]:
@env.task
async def route_request(request: str) -> str:
    """Route a single request: call the LLM router, then dispatch to the right handler."""
    decision = await router_node(request=request)
    if decision == "booker":
        return await booking_handler(request=request)
    elif decision == "info":
        return await info_handler(request=request)
    else:
        return await unclear_handler(request=request)

7. Run the task:

In [ ]:
run = flyte.run(route_request, request="Find tickets to Amsterdam")
run.wait()
print(run.outputs()[0])


### Remote execution

[**Optional**] Configure your connection to a remote Flyte cluster. This tells Flyte where to run your workflows and how to build container images.

**Configuration Options:**
- `endpoint`: Your Flyte cluster URL
- `org`: Your organization name
- `project`: Project to organize workflows
- `domain`: Environment (development, staging, production)
- `image_builder`: Use "remote" to build images on the cluster (no local Docker required)

In [ ]:
run = flyte.run(route_request, request="What is the capital of France?")
run.wait()
print(run.outputs()[0])


## Scaling the pattern

1. Make it batchable for multiple requests:

In [ ]:
@env.task
async def route_requests_batch(requests: list[str]) -> list[str]:
    """Route a batch of requests in parallel with bounded concurrency."""
    sem = asyncio.Semaphore(20)

    async def _one(req: str) -> str:
        async with sem:
            return await route_request(req)

    tasks = [_one(r) for r in requests]
    return list(await asyncio.gather(*tasks))